# TFM — Evaluación cuantitativa del sistema (Corpus GAJJIA)

Este notebook ejecuta el **pipeline completo de DocAudit Agent** sobre un corpus anotado y calcula métricas cuantitativas a nivel de campo.

Métricas reportadas:
- F1 a nivel de campo (micro)
- Exact match rate a nivel de campo
- Latencia por documento (seg)
- Pico de RAM del proceso (GB)

Requisitos:
- Ollama corriendo (http://localhost:11434)
- Modelos: `llama3.2:3b` (texto), `qwen2.5vl:7b` (visión para PDFs escaneados)


In [ ]:
import os
import sys
import json
import time
import re
import statistics
import threading
from pathlib import Path
from typing import Any

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "core").exists() and (p / "agents").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import psutil

from core.document_loader import extract_text_from_pdf_bytes, extract_text_from_scanned_pdf_bytes
from core.schema_models import DocSchema, SchemaField
import agents.extractor as extractor_mod
from agents.extractor import extract_from_text
from core.normalizer import normalize_extracted
from core.validator import validate_extracted
from agents.auditor import audit_document

print("OK: imports")

In [ ]:
CORPUS_DIR = Path(os.getenv("DOCAUDIT_CORPUS_DIR", r"c:\Users\gusta\Desktop\Maestria\0. TFM\DocAudit Agent\Corpus_GAJJIA\Corpus_GAJJIA"))
USE_VISION_IF_NO_TEXT = True
USE_RAG_EVIDENCE = False

TARGET_F1 = 0.85
TARGET_EXACT = 0.75
TARGET_LAT_S = 30.0
TARGET_RAM_GB = 16.0

pdf_paths = sorted(CORPUS_DIR.glob("*.pdf"))
json_paths = sorted(CORPUS_DIR.glob("*.json"))

pdf_stems = {p.stem for p in pdf_paths}
json_stems = {p.stem for p in json_paths}
paired = sorted([p for p in pdf_paths if p.stem in json_stems])
unpaired_pdfs = sorted([p.name for p in pdf_paths if p.stem not in json_stems])
unpaired_json = sorted([p.name for p in json_paths if p.stem not in pdf_stems])

print("Corpus:", str(CORPUS_DIR))
print("PDFs:", len(pdf_paths), "| JSONs:", len(json_paths), "| Pares PDF+JSON:", len(paired))
if unpaired_pdfs:
    print("PDFs sin GT (mismo nombre .json):", unpaired_pdfs[:10])
if unpaired_json:
    print("GT sin PDF (mismo nombre .pdf):", unpaired_json[:10])

In [ ]:
if not USE_RAG_EVIDENCE:
    import core.rag as rag

    def _no_rag(queries: list[str], chunks: list[Any], top_k: int = 1, doc_id: str | None = None):
        return [[] for _ in queries]

    rag.retrieve_best_evidence_batch = _no_rag
    extractor_mod.retrieve_best_evidence_batch = _no_rag
    print("RAG (evidencias) desactivado")
else:
    print("RAG (evidencias) activado")

In [ ]:
def guess_doc_family(filename: str) -> str:
    f = (filename or "").lower()
    if "irpf" in f or "modelo_100" in f or "modelo100" in f:
        return "IRPF"
    if "factura" in f:
        return "FACTURA"
    if "extracto" in f or "banco" in f:
        return "EXTRACTO"
    if "contrato" in f:
        return "CONTRATO"
    return "OTRO"


def guess_type(value: Any) -> str:
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int) and not isinstance(value, bool):
        return "integer"
    if isinstance(value, float):
        return "number"
    if isinstance(value, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", value.strip()):
        return "date"
    return "string"


def build_schema_from_ground_truth(gt: dict[str, Any], *, name: str) -> DocSchema:
    fields: list[SchemaField] = []
    for k, v in gt.items():
        if k == "id_documento":
            continue
        fields.append(SchemaField(name=str(k), type=guess_type(v), required=False, description=str(k)))
    return DocSchema(name=name, version="1.0", fields=fields)


def normalize_for_match(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value).strip().lower()
    return str(value).strip().lower()


def field_level_counts(y_true: dict[str, Any], y_pred: dict[str, Any]) -> dict[str, int]:
    keys = set(y_true.keys()) | set(y_pred.keys())
    tp = fp = fn = 0
    for k in keys:
        real = normalize_for_match(y_true.get(k))
        pred = normalize_for_match(y_pred.get(k))
        if real and pred and real == pred:
            tp += 1
        elif pred and real != pred:
            fp += 1
        elif real and not pred:
            fn += 1
    return {"tp": tp, "fp": fp, "fn": fn, "n": len(keys)}


def f1_from_counts(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return f1, precision, recall


def run_with_ram_peak(fn):
    proc = psutil.Process(os.getpid())
    peak = 0
    running = True

    def monitor():
        nonlocal peak, running
        while running:
            rss = proc.memory_info().rss
            if rss > peak:
                peak = rss
            time.sleep(0.05)

    t = threading.Thread(target=monitor, daemon=True)
    t.start()
    try:
        start = time.perf_counter()
        out = fn()
        elapsed = time.perf_counter() - start
        return out, elapsed, peak
    finally:
        running = False
        t.join(timeout=1.0)


In [ ]:
def load_ground_truth(pdf_path: Path) -> dict[str, Any]:
    gt_path = pdf_path.with_suffix(".json")
    return json.loads(gt_path.read_text(encoding="utf-8"))


def extract_text_from_pdf(pdf_path: Path) -> dict[str, Any]:
    pdf_bytes = pdf_path.read_bytes()
    t0 = time.perf_counter()
    extracted = extract_text_from_pdf_bytes(pdf_bytes)
    t1 = time.perf_counter()
    text = (extracted.get("text") or "").strip()
    pages = extracted.get("page_texts")
    method = extracted.get("method")

    t2 = t1
    if USE_VISION_IF_NO_TEXT and not text:
        extracted_v = extract_text_from_scanned_pdf_bytes(pdf_bytes)
        t2 = time.perf_counter()
        text = (extracted_v.get("text") or "").strip()
        pages = extracted_v.get("page_texts")
        method = "vision"

    return {
        "text": text,
        "pages": pages,
        "method": method,
        "t_text_s": round(t1 - t0, 3),
        "t_vision_s": round((t2 - t1) if t2 != t1 else 0.0, 3),
    }


def run_pipeline_for_gt(text: str, pages: list[str] | None, gt: dict[str, Any], name: str) -> dict[str, Any]:
    schema = build_schema_from_ground_truth(gt, name=name)
    raw = extract_from_text(text, schema, pages=pages, doc_id=None)
    if isinstance(raw, dict) and "fields" in raw and "details" in raw:
        fields = raw.get("fields") or {}
        details = raw.get("details") or {}
    else:
        fields = raw
        details = {}
    norm = normalize_extracted(fields, schema)
    val = validate_extracted(norm["normalized"], schema)
    rep = audit_document(schema, norm["normalized"], val, field_details=details)
    return {
        "schema": {"name": schema.name, "version": schema.version},
        "extracted": norm["normalized"],
        "validation": val,
        "report": rep,
    }


def process_one(pdf_path: Path) -> dict[str, Any]:
    gt = load_ground_truth(pdf_path)
    family = guess_doc_family(pdf_path.name)
    gt_clean = {k: v for k, v in gt.items() if k != "id_documento"}

    text_info = extract_text_from_pdf(pdf_path)
    text = text_info["text"]
    pages = text_info["pages"] if isinstance(text_info["pages"], list) else None

    def _run():
        return run_pipeline_for_gt(text, pages, gt, name=f"gt_{pdf_path.stem}")

    result, t_total, peak_bytes = run_with_ram_peak(_run)
    counts = field_level_counts(gt_clean, result.get("extracted") or {})
    f1, precision, recall = f1_from_counts(counts["tp"], counts["fp"], counts["fn"])
    exact = (counts["tp"] / counts["n"]) if counts["n"] else 0.0

    return {
        "file": pdf_path.name,
        "family": family,
        "method": text_info.get("method"),
        "t_text_s": text_info.get("t_text_s"),
        "t_vision_s": text_info.get("t_vision_s"),
        "latency_s": round(t_total, 3),
        "ram_peak_gb": round(peak_bytes / (1024**3), 3),
        "tp": counts["tp"],
        "fp": counts["fp"],
        "fn": counts["fn"],
        "fields_total": counts["n"],
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "exact_match_rate": round(exact, 3),
        "result": result,
        "ground_truth": gt_clean,
    }


if paired:
    sample = process_one(paired[0])
    print({k: sample[k] for k in sample.keys() if k not in {"result", "ground_truth"}})
else:
    print("No hay pares PDF+JSON para evaluar.")


In [ ]:
results: list[dict[str, Any]] = []
for p in paired:
    print("Procesando:", p.name)
    results.append(process_one(p))

def summarize(rows: list[dict[str, Any]]) -> dict[str, Any]:
    if not rows:
        return {}
    tp = sum(r["tp"] for r in rows)
    fp = sum(r["fp"] for r in rows)
    fn = sum(r["fn"] for r in rows)
    f1, precision, recall = f1_from_counts(tp, fp, fn)
    exact = sum(r["tp"] for r in rows) / sum(r["fields_total"] for r in rows) if sum(r["fields_total"] for r in rows) else 0.0
    lat = [r["latency_s"] for r in rows]
    ram = [r["ram_peak_gb"] for r in rows]
    return {
        "docs": len(rows),
        "f1_micro": round(f1, 3),
        "precision_micro": round(precision, 3),
        "recall_micro": round(recall, 3),
        "exact_match_rate": round(exact, 3),
        "lat_avg_s": round(statistics.mean(lat), 3),
        "lat_p50_s": round(statistics.median(lat), 3),
        "lat_p95_s": round(sorted(lat)[max(0, int(0.95 * (len(lat)-1)))], 3) if len(lat) > 1 else round(lat[0], 3),
        "ram_avg_gb": round(statistics.mean(ram), 3),
        "ram_max_gb": round(max(ram), 3),
    }

overall = summarize(results)
print("\n=== Resumen global ===")
print(overall)

by_family: dict[str, list[dict[str, Any]]] = {}
for r in results:
    by_family.setdefault(r["family"], []).append(r)

print("\n=== Resumen por familia ===")
for fam in sorted(by_family.keys()):
    print(fam, summarize(by_family[fam]))


In [ ]:
def pass_fail(summary: dict[str, Any]) -> dict[str, Any]:
    if not summary:
        return {}
    return {
        "F1_micro": {"valor": summary.get("f1_micro"), "meta": TARGET_F1, "cumple": float(summary.get("f1_micro") or 0) >= TARGET_F1},
        "Exact_match": {"valor": summary.get("exact_match_rate"), "meta": TARGET_EXACT, "cumple": float(summary.get("exact_match_rate") or 0) >= TARGET_EXACT},
        "Lat_avg_s": {"valor": summary.get("lat_avg_s"), "meta": TARGET_LAT_S, "cumple": float(summary.get("lat_avg_s") or 0) <= TARGET_LAT_S},
        "RAM_max_gb": {"valor": summary.get("ram_max_gb"), "meta": TARGET_RAM_GB, "cumple": float(summary.get("ram_max_gb") or 0) <= TARGET_RAM_GB},
    }

print("\n=== Chequeo de metas (global) ===")
print(pass_fail(overall))